# 
Treasure Hunt Game Notebook

## Read and Review Your Starter Code
The theme of this project is a popular treasure hunt game in which the player needs to find the treasure before the pirate does. While you will not be developing the entire game, you will write the part of the game that represents the intelligent agent, which is a pirate in this case. The pirate will try to find the optimal path to the treasure using deep Q-learning. 

In [1]:
import py_compile
try:
    py_compile.compile("GameExperience.py", doraise=True)
    print("GameExperience.py compiles cleanly")
except Exception as e:
    print("Still broken:", e)


GameExperience.py compiles cleanly


<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Use this <b>TreasureHuntGame_starterCode.ipynb</b> file to complete your assignment. 
<br><br>
You have been provided with two Python classes and this notebook to help you with this assignment. The first class, <b>TreasureMaze.py</b>, represents the environment, which includes a maze object defined as a matrix. The second class, <b>GameExperience.py</b>, stores the episodes – that is, all the states that come in between the initial state and the terminal state. This is later used by the agent for learning by experience, called "exploration". This notebook shows how to play a game. Your task is to complete the deep Q-learning implementation in the qtrain() function for which a skeleton implementation has been provided. 
</div>
<br>
<div class="alert alert-block alert-info" style="color:black;">
<b>NOTE: </b>The code block you will need to complete will have <b>#TODO</b> as a header.
<br> First, read and review the next few code and instruction blocks to understand the code that you have been given.</div>

<div class="alert alert-block alert-warning" style="color: #333333;">
<b>Installations</b> The following command will install the necessary Python libraries to necessary to run this application. If you see a "[notice] A new release of pip is available: 23.1.2 -> 25.2" at the end of the installation, you may disregard that statement. 
</div>

In [2]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


<h2>Tensorflow CPU Acceleration Warning</h2>
<div class="alert alert-block alert-danger" style="color: #333333;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
    <b>Example messages:</b>
    <ul>
        <li>oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders</li>
        <li>WARNING: All log messages before absl::InitializeLog() is called are written to STDERR</li>
</div>

In [11]:
from __future__ import print_function
import os, sys, time, datetime, json, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import clone_model
from keras.models import Sequential
from keras.layers import Dense, Activation, PReLU
from keras.optimizers import SGD , Adam, RMSprop
import matplotlib.pyplot as plt
from TreasureMaze import TreasureMaze
from GameExperience import GameExperience
%matplotlib inline

<h2> Maze Object Generation</h2>

<div class="alert alert-block alert-info" style="color:black;">
    <b>NOTE:</b>  The following code block contains an 8x8 matrix that will be used as a maze object:
</div>

In [12]:
import numpy as np
maze = np.array([
    [ 1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  0.,  1.,  1.,  1.,  0.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  0.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  0.,  0.,  0.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  1.,  1.]
])

<h2>Helper Functions and Global Variables</h2>

<div class="alert alert-block alert-info" style="color:black;">
This <b>show()</b> helper function allows a visual representation of the maze object:
</div>

In [14]:
def show(qmaze):
    plt.grid('on')
    nrows, ncols = qmaze.maze.shape
    ax = plt.gca()
    ax.set_xticks(np.arange(0.5, nrows, 1))
    ax.set_yticks(np.arange(0.5, ncols, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    canvas = np.copy(qmaze.maze)
    for row,col in qmaze.visited:
        canvas[row,col] = 0.6
    pirate_row, pirate_col, _ = qmaze.state
    canvas[pirate_row, pirate_col] = 0.3   # pirate cell
    canvas[nrows-1, ncols-1] = 0.9 # treasure cell
    img = plt.imshow(canvas, interpolation='none', cmap='gray')
    return img

The <b>pirate agent</b> can move in four directions: left, right, up, and down. 

<div class="alert alert-block alert-warning" style="color:black;">
<b>Note:</b> While the agent primarily learns by experience through exploitation, often, the agent can choose to explore the environment to find previously undiscovered paths. This is called "exploration" and is defined by epsilon. This value is the <b>EXPLORATION</b> values from the Cartpole assignment. The hyperparameters are provided here and used in the <b>qtrain()</b> method. 
You are encouraged to try various values for the exploration factor and see how the algorithm performs.
</div>

In [13]:
LEFT = 0
UP = 1
RIGHT = 2
DOWN = 3


# Exploration factor
epsilon = 1.0
epsilon_min = 0.05
epsilon_decaye = 0.99
patience = 10

# Actions dictionary
actions_dict = {
    LEFT: 'left',
    UP: 'up',
    RIGHT: 'right',
    DOWN: 'down',
}

num_actions = len(actions_dict)


The sample code block and output below show creating a maze object and performing one action (DOWN), which returns the reward. The resulting updated environment is visualized.

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, PReLU

import numpy as np

num_actions = 4

maze = np.array([
    [1., 0., 1., 1., 1., 1., 1., 1.],
    [1., 0., 1., 1., 1., 0., 1., 1.],
    [1., 1., 1., 1., 0., 1., 0., 1.],
    [1., 1., 1., 0., 1., 1., 1., 1.],
    [1., 1., 0., 1., 1., 1., 1., 1.],
    [1., 1., 1., 0., 1., 0., 0., 0.],
    [1., 1., 1., 0., 1., 1., 1., 1.],
    [1., 1., 1., 1., 0., 1., 1., 1.]
])

def build_model(maze):
    model = Sequential()
    model.add(Dense(maze.size, input_shape=(maze.size,)))
    model.add(PReLU())
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(num_actions))
    model.compile(
        optimizer='adam',
        run_eagerly=True,
        loss='mse',
        jit_compile=False
    )
    return model

model = build_model(maze)
# model.load_weights("pirate.weights.h5")
print("Fresh model built")


Fresh model built


<div class="alert alert-block alert-warning" style="color:black;">
    <b>NOTE:</b> This <b>play_game()</b> function simulates a full game based on the provided trained model. The other parameters include the TreasureMaze object, the starting position of the pirate and max amount of steps to make sure the code does not get stuck in a loop.
</div>

In [15]:
qmaze = TreasureMaze(maze)


<div class="alert alert-block alert-warning" style="color:black;">
<b>Note: </b>
    This <b>completion_check()</b> function helps you to determine whether the pirate can win any game at all. If your maze is not well designed, the pirate may not win any game at all. In this case, your training would not yield any result. The provided maze in this notebook ensures that there is a path to win and you can run this method to check.
</div>

In [18]:
def completion_check(model, maze_or_qmaze, max_steps=None):
    # Accept either raw numpy maze or TreasureMaze instance
    if isinstance(maze_or_qmaze, TreasureMaze):
        qmaze = maze_or_qmaze
    else:
        qmaze = TreasureMaze(maze_or_qmaze)

    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            continue
        if not play_game(model, qmaze, cell, max_steps=max_steps):
            return False
    return True

<div class="alert alert-block alert-warning" style="color:black;">
<b>Note: </b>
</b>The <b>build_model()</b> function in the block below will build the neural network model. Review the code and note the number of layers, as well as the activation, optimizer, and loss functions that are used to train the model.
</div>

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b>
    This <b>train_step()</b> helper function in the block below is used to help predict Q-values (quality values) in the current modelto see how good each action is in a given state and improve the Q-network by reducing the gap between what is predicted and what should have been predicted. 
</div>
<br>
<div class="alert alert-block alert-info" style="color:black;">
If you're interested in reading up on the <i>@tf.function</i>, which is a decorator for Tensorflow to run this code into a TensorFlow computation graph, please refer to this link: <a href="https://www.tensorflow.org/guide/intro_to_graphs">https://www.tensorflow.org/guide/intro_to_graphs</a>
</div>


In [ ]:
def completion_check(model, maze_or_qmaze, max_steps=64):
    if isinstance(maze_or_qmaze, TreasureMaze):
        qmaze = maze_or_qmaze
    else:
        qmaze = TreasureMaze(maze_or_qmaze)

    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            continue

        if not play_game(model, qmaze, cell, max_steps=max_steps):
            return False

    return True

print("Completion check:", completion_check(model, qmaze, max_steps=64))

In [17]:
def play_game(model, qmaze, pirate_cell, max_steps=None):
    """Play one game from `pirate_cell`.

    Runs the trained model greedily (no exploration) and returns True when the
    treasure is reached, False otherwise. max_steps guards against a game that
    never terminates.
    """
    qmaze.reset(pirate_cell)
    envstate = qmaze.observe()
    status = "not_over"
    steps = 0

    while status == "not_over":
        if max_steps is not None and steps >= max_steps:
            return False

        valid_actions = qmaze.valid_actions()
        if not valid_actions:
            return False

        # Greedy action selection: take the move with the highest Q-value.
        state = np.asarray(envstate, dtype=np.float32).reshape(1, -1)
        q_values = model(state, training=False).numpy()[0]
        action = max(valid_actions, key=lambda a: q_values[a])

        envstate, reward, status = qmaze.act(action)
        steps += 1

    return status == "win"


<h2>Tensorflow GPU Warning</h2>
<div class="alert alert-block alert-danger" style="color: #333333;">
    You will see a <b>warning in red</b> "INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.". This is simply coming from <b>Tensorflow skipping using GPU for this assignment.</b>  
</div>

In [ ]:
import tensorflow as tf
loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam()

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        q_values = model(x, training=True)
        loss = loss_fn(y, q_values)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

# #TODO: Complete the Q-Training Algorithm Code Block

<div class="alert alert-block alert-info" style="color:black;">
    This is your deep Q-learning implementation. The goal of your deep Q-learning implementation is to find the best possible navigation sequence that results in reaching the treasure cell while maximizing the reward. In your implementation, you need to determine the optimal number of epochs to achieve a 100% win rate.
</div>
    <b>Pseudocode:</b>
    <br>
    For each epoch:
        Reset the environment at a random starting cell
        agent_cell = randomly select a free cell
        <br>
        <b>Hint:</b> Review the reset method in the TreasureMaze.py class.
    
        Set the initial environment state
        env_state should reference the environment's current state
        Hint: Review the observe method in the TreasureMaze.py class.

        While game status is not game over:
           previous_envstate = env_state
            Decide on an action:
                - If possible, take a random valid exploration action and 
                  randomly choose action (left, right, up, down)
                  and assign it to an action variable
                - Else, pick the best exploitation action from the model and assign it to an action variable
                  Hint: Review the predict method in the GameExperience.py class.
    
           Retrieve the values below from the act() method.
           env_state, reward, game_status = qmaze.act(action)
           Hint: Review the act method in the TreasureMaze.py class.
    
            Track the wins and losses from the game_status using win_history 
         
           Store the episode below in the Experience replay object
           episode = [previous_envstate, action, reward, envstate, game_status]
           Hint: Review the remember method in the GameExperience.py class.
        
           Train neural network model and evaluate loss
           Hint: Call GameExperience.get_data to retrieve training data (input and target) 
           and pass to the train_step method and assign it to batch_loss and append to the loss variable
        
      If the win rate is above the threshold and your model passes the completion check, that would be your epoch.

Note: A 100% win rate <b>DOES NOT EXPLICITLY MEAN</b> that you have solved the maze. It simply indicates that during the last evaluation, the pirate <i>happened</i> to get to the treasure. Be sure to utilise the <b>completion_check()</b> function to validate your pirate found the treasure at every starting point and consistently! 

<b> You will need to complete the section starting with #START_HERE. Please use the pseudocode above as guidance. </b>


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.99
patience = 10


In [22]:
# =====================================================================
# CS 370 Project Two - Q-Training Algorithm (deep Q-learning)
# SELF-CONTAINED version: defines its own hyperparameters and helper
# functions, so it works no matter which other cells exist or in what
# order they run. Paste this whole cell into TreasureHuntGame.ipynb,
# replacing the #TODO cell. TreasureMaze.py and GameExperience.py are
# left untouched, as the guidelines require.
# =====================================================================

import datetime
import random

import numpy as np

from TreasureMaze import TreasureMaze
from GameExperience import GameExperience
from tensorflow.keras.models import clone_model


def format_time(seconds):
    """Present an elapsed time in a readable form."""
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return "{}h {}m {}s".format(hours, minutes, secs)


# ---------------------------------------------------------------
# Hyperparameters. Defined here rather than relying on a separate
# cell, so qtrain() always has them.
# ---------------------------------------------------------------
epsilon = 1.0          # exploration rate: 1.0 = fully random at first
epsilon_min = 0.05     # floor, so a little exploration always remains
epsilon_decay = 0.99   # per-epoch decay toward that floor
patience = 10


def qtrain(model, maze, **opt):
    """Train the pirate agent with deep Q-learning.

    Every episode starts at the top-left cell (0, 0), which is where the
    specification places the pirate. Starting from the required position means
    the win rate reflects the agent's ability to solve the maze from the start
    rather than from whichever cell happens to sit beside the treasure.

    On each step the agent explores (random legal move) with probability
    epsilon, otherwise exploits the move with the highest predicted Q-value.
    Transitions are stored in replay memory and the network is trained on
    batches sampled from it after each game. Returns the epoch at which a
    100% win rate was reached.
    """
    global epsilon

    # ---- Hyperparameters, overridable by the caller ----
    n_epoch = opt.get('n_epoch', 15000)            # maximum training games
    max_memory = opt.get('max_memory', 1000)       # replay buffer capacity
    data_size = opt.get('data_size', 50)           # transitions per batch
    target_update_freq = opt.get('target_update_freq', 50)  # sync interval
    train_batches = opt.get('train_batches', 4)    # learning passes per game

    # Local defaults, so this function works even if the module-level
    # hyperparameters above were overwritten elsewhere in the notebook.
    eps_decay = opt.get('epsilon_decay', epsilon_decay)
    eps_min = opt.get('epsilon_min', epsilon_min)

    start_time = datetime.datetime.now()
    qmaze = TreasureMaze(maze)

    # A separate target network supplies the training targets. Holding its
    # weights still between syncs keeps those targets from chasing the
    # network's own latest update, which would destabilise training.
    target_model = clone_model(model)
    target_model.set_weights(model.get_weights())
    experience = GameExperience(model, target_model, max_memory=max_memory)

    win_history = []                 # 1 for a won game, 0 otherwise
    hsize = qmaze.maze.size // 2     # rolling window used for the win rate
    win_rate = 0.0
    loss = 0.0
    epochs_needed = n_epoch

    # SPEED: call the model directly instead of through GameExperience.predict.
    # predict() routes a single state through Keras' Python prediction path on
    # every step, which retraces and dominates the runtime. A direct call with
    # a reused batch of one row uses the same traced graph throughout.
    action_buf = np.zeros((1, maze.size), dtype=np.float32)

    for epoch in range(n_epoch):
        # ---- Play one episode from the required top-left start ----
        agent_cell = (0, 0)
        qmaze.reset(agent_cell)
        envstate = qmaze.observe()
        game_status = "not_over"

        while game_status == "not_over":
            previous_envstate = envstate
            valid_actions = qmaze.valid_actions()

            # Exploration versus exploitation: with probability epsilon take a
            # random legal move to gather new experience, otherwise take the
            # action the network currently rates most highly.
            if np.random.rand() < epsilon:
                action = random.choice(valid_actions)
            else:
                action_buf[0, :] = np.asarray(envstate, dtype=np.float32)
                q_values = model(action_buf, training=False).numpy()[0]
                action = max(valid_actions, key=lambda a: q_values[a])

            # Apply the action and record the resulting transition.
            envstate, reward, game_status = qmaze.act(action)
            done = game_status != "not_over"
            experience.remember(
                [previous_envstate, action, reward, envstate, done])

        win_history.append(1 if game_status == "win" else 0)

        # ---- Learn from replay memory ----
        # Sampling several batches per episode reuses stored transitions and
        # breaks the correlation between consecutive steps of one game.
        for _ in range(train_batches):
            inputs, targets = experience.get_data(batch_size=data_size)
            loss = model.train_on_batch(inputs, targets)

        # ---- Periodically refresh the target network ----
        if (epoch + 1) % target_update_freq == 0:
            target_model.set_weights(model.get_weights())

        # ---- Rolling win rate over the most recent games ----
        win_rate = sum(win_history[-hsize:]) / len(win_history[-hsize:])

        # Decay exploration toward its floor, so behaviour shifts from
        # discovery early in training to reliance on what was learned.
        epsilon = max(eps_min, epsilon * eps_decay)

        if (epoch + 1) % 10 == 0:
            print("Epoch: {}, Loss: {:.4f}, Win Rate: {:.2f}".format(
                epoch + 1, float(loss), win_rate))

        # ---- Confirm a genuine solution, then stop ----
        # completion_check() verifies that the treasure can be reached from
        # every free cell, so it is a stronger test than the rolling win rate.
        # It runs on every epoch past the warm-up window, so a real solution
        # ends training promptly.
        if len(win_history) >= hsize and win_rate == 1.0:
            if completion_check(model, qmaze, max_steps=100):
                epochs_needed = epoch + 1
                print("Reached 100% win rate at epoch:", epochs_needed)
                break

    print("Training time:",
          format_time((datetime.datetime.now() - start_time).total_seconds()))
    return epochs_needed


## Test Your Model

Now we will start testing the deep Q-learning implementation. To begin, select **Cell**, then **Run All** from the menu bar. This will run your notebook. As it runs, you should see output begin to appear beneath the next few cells. The code below creates an <b>instance</b> of TreasureMaze. This does not show your actual training done.

In [ ]:
qmaze = TreasureMaze(maze)
show(qmaze)

In the next code block, you will build your model using the <b>build_model</b> function and train it using deep Q-learning. Note: This step takes several minutes to fully run.



<div class="alert alert-block alert-danger" style="color: #333333;">
  <b>WARNING</b>  If you did not attempt the assignment, the code <b>will</b> error out at this section.
 </div>

In [ ]:
  epsilon = 1.0
  model = build_model(maze)
  model.compile(optimizer='adam', loss='mse', run_eagerly=False)
  epochs_needed = qtrain(model, maze, n_epoch=1000, max_memory=8*maze.size, data_size=32, target_update_freq=10)
  print("Epochs needed:", epochs_needed)
  model.save_weights("pirate.weights.h5")

Epoch: 10, Loss: 0.0022, Win Rate: 1.00
Epoch: 20, Loss: 0.0027, Win Rate: 1.00
Epoch: 30, Loss: 0.0026, Win Rate: 1.00
Epoch: 40, Loss: 0.0027, Win Rate: 1.00
Epoch: 50, Loss: 0.0026, Win Rate: 1.00
Epoch: 60, Loss: 0.0025, Win Rate: 1.00


<div class="alert alert-block alert-warning" style="color:black;">
<b>Note: </b> This cell will check to see if the model passes the completion check. Note: This could take several minutes.
</div>

In [25]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    run_eagerly=False,
    jit_compile=False
)

print("Faster training mode enabled!")

Faster training mode enabled!


In [ ]:
print("Completion check:", completion_check(model, qmaze))
show(qmaze)

In [26]:
epsilon = 0.30
epsilon_min = 0.05
epsilon_decay = 0.995

print("Exploration settings updated successfully!")

Exploration settings updated successfully!


This cell will test your model for one game. It will start the pirate at the top-left corner and run <b>play_game()</b>. The agent should find a path from the starting position to the target (treasure). The treasure is located in the bottom-right corner.

In [29]:
epochs_completed = qtrain(
    model,
    maze,
    n_epoch=20,
    max_memory=1000,
    data_size=32,
    target_update_freq=20
)

print("Training test completed!")

KeyboardInterrupt: 

In [ ]:
model = build_model(maze)
model.load_weights("pirate.weights.h5")
print("Model weights restored successfully!")

In [ ]:
pirate_start = (0, 0)
play_game(model, qmaze, pirate_start)
show(qmaze)




## Save and Submit Your Work

<div class="alert alert-block alert-info" style="color:black;">
    <b>Hint:</b> To use the markdown block below, double click in the <b>Type Markdown and LaTeX:  𝛼2</b> block below, to turn it back to html, Run the cell.
</div>

After you have finished creating the code for your notebook, save your work.
Make sure that your notebook contains your name in the filename (e.g. Doe_Jane_ProjectTwo.html). Download this file as an .html file clicking on ***file*** in *Jupyter Notebook*, navigating down to ***Download as*** and clicking on ***.html***. 
Download a copy of your .html file and submit it to Brightspace.

In [ ]:
import numpy as np

model = build_model(maze)

test_inputs = np.zeros((2, maze.size), dtype=np.float32)
test_targets = np.zeros((2, num_actions), dtype=np.float32)

with tf.device('/CPU:0'):
    test_loss = model.train_on_batch(test_inputs, test_targets)

print("Training batch successful:", test_loss)

In [ ]:
print("Game result:", play_game(model, qmaze, (0, 0)))